In [3]:
import duckdb
conn = duckdb.connect("butternutbox.db")

In [4]:
conn.execute("""
    SELECT
        (SELECT COUNT(*) FROM customers)     AS customers,
        (SELECT COUNT(*) FROM subscriptions) AS subscriptions,
        (SELECT COUNT(*) FROM orders)        AS orders,
        (SELECT COUNT(*) FROM cancellations) AS cancellations
""").df()

,customers,subscriptions,orders,cancellations
0,40,40,348,11


In [5]:
conn.execute("""
    SELECT
        DATE_TRUNC('month', signup_date) AS signup_month,
        COUNT(*)                          AS new_customers
    FROM customers
    GROUP BY signup_month
    ORDER BY signup_month
""").df()

,signup_month,new_customers
0,2023-01-01,6
1,2023-02-01,5
2,2023-03-01,4
3,2023-04-01,6
4,2023-05-01,3
5,2023-06-01,4
6,2023-07-01,3
7,2023-08-01,3
8,2023-09-01,3
9,2023-10-01,2


In [6]:
conn.execute("""
    SELECT
        DATE_TRUNC('month', signup_date) AS signup_month,
        COUNT(*)                          AS new_customers,
        SUM(COUNT(*)) OVER (ORDER BY DATE_TRUNC('month', signup_date)) AS total_customers
    FROM customers
    GROUP BY signup_month
    ORDER BY signup_month
""").df()

,signup_month,new_customers,total_customers
0,2023-01-01,6,6.0
1,2023-02-01,5,11.0
2,2023-03-01,4,15.0
3,2023-04-01,6,21.0
4,2023-05-01,3,24.0
5,2023-06-01,4,28.0
6,2023-07-01,3,31.0
7,2023-08-01,3,34.0
8,2023-09-01,3,37.0
9,2023-10-01,2,39.0


In [7]:
conn.execute("""
    SELECT
        DATE_TRUNC('month', c.signup_date) AS signup_month,
        s.plan_name,
        COUNT(*)                           AS new_customers
    FROM customers c
    JOIN subscriptions s ON c.customer_id = s.customer_id
    GROUP BY signup_month, s.plan_name
    ORDER BY signup_month, s.plan_name
""").df()

,signup_month,plan_name,new_customers
0,2023-01-01,Full,2
1,2023-01-01,Regular,2
2,2023-01-01,Starter,2
3,2023-02-01,Full,2
4,2023-02-01,Regular,2
5,2023-02-01,Starter,1
6,2023-03-01,Full,1
7,2023-03-01,Regular,1
8,2023-03-01,Starter,2
9,2023-04-01,Full,2


In [8]:
conn.execute("""
    SELECT
        DATE_TRUNC('month', c.signup_date) AS signup_month,
        COUNT(*) FILTER (WHERE s.plan_name = 'Starter') AS starter,
        COUNT(*) FILTER (WHERE s.plan_name = 'Regular') AS regular,
        COUNT(*) FILTER (WHERE s.plan_name = 'Full')    AS full
    FROM customers c
    JOIN subscriptions s ON c.customer_id = s.customer_id
    GROUP BY signup_month
    ORDER BY signup_month
""").df()

,signup_month,starter,regular,full
0,2023-01-01,2,2,2
1,2023-02-01,1,2,2
2,2023-03-01,2,1,1
3,2023-04-01,2,2,2
4,2023-05-01,1,1,1
5,2023-06-01,1,2,1
6,2023-07-01,1,1,1
7,2023-08-01,1,1,1
8,2023-09-01,1,1,1
9,2023-10-01,0,1,1


In [9]:
conn.execute("""
    SELECT
        s.plan_name,
        COUNT(DISTINCT o.subscription_id)   AS active_subscriptions,
        SUM(o.order_value)                  AS total_revenue,
        ROUND(SUM(o.order_value) * 100.0 / 
            SUM(SUM(o.order_value)) OVER (), 1) AS revenue_pct
    FROM orders o
    JOIN subscriptions s ON o.subscription_id = s.subscription_id
    GROUP BY s.plan_name
    ORDER BY total_revenue DESC
""").df()

,plan_name,active_subscriptions,total_revenue,revenue_pct
0,Full,14,24087.0,43.6
1,Regular,14,19304.0,34.9
2,Starter,12,11880.0,21.5


In [10]:
conn.execute("""
    SELECT
        c.first_name,
        c.last_name,
        c.dog_size,
        s.plan_name,
        s.start_date,
        cn.cancellation_date,
        cn.reason,
        DATE_DIFF('day', s.start_date, cn.cancellation_date) AS days_active
    FROM cancellations cn
    JOIN subscriptions s ON cn.subscription_id = s.subscription_id
    JOIN customers c ON s.customer_id = c.customer_id
    ORDER BY days_active
""").df()

,first_name,last_name,dog_size,plan_name,start_date,cancellation_date,reason,days_active
0,Sophie,Brown,small,Starter,2023-01-20,2023-02-19,Too expensive,30
1,Reuben,Hernandez,large,Full,2023-01-30,2023-03-11,Moving abroad,40
2,Isla,Jones,small,Starter,2023-04-18,2023-06-02,Dog did not like the food,45
3,Freddie,Lewis,medium,Regular,2023-06-18,2023-08-02,Financial reasons,45
4,Lily,Anderson,small,Starter,2023-08-14,2023-10-13,Financial reasons,60
5,Emily,Harris,medium,Regular,2023-02-10,2023-04-11,Too expensive,60
6,Evie,Robinson,medium,Regular,2023-04-22,2023-06-21,Switched to another brand,60
7,Arlo,Hill,large,Full,2023-04-08,2023-06-07,Too expensive,60
8,Monty,Nelson,large,Full,2023-09-05,2023-11-04,Financial reasons,60
9,Penelope,Carter,large,Full,2023-10-15,2023-12-14,Too expensive,60


In [11]:
conn.execute("""
    SELECT
        s.plan_name,
        COUNT(DISTINCT s.subscription_id)                     AS total_customers,
        COUNT(DISTINCT cn.subscription_id)                    AS churned,
        ROUND(COUNT(DISTINCT cn.subscription_id) * 100.0 /
              COUNT(DISTINCT s.subscription_id), 1)           AS churn_rate_pct
    FROM subscriptions s
    LEFT JOIN cancellations cn ON s.subscription_id = cn.subscription_id
    GROUP BY s.plan_name
    ORDER BY churn_rate_pct DESC
""").df()

,plan_name,total_customers,churned,churn_rate_pct
0,Full,14,5,35.7
1,Starter,12,3,25.0
2,Regular,14,3,21.4


In [12]:
conn.execute("""
    SELECT
        s.plan_name,
        ROUND(AVG(DATE_DIFF('day', s.start_date, cn.cancellation_date)), 0) AS avg_days_before_cancel,
        MIN(DATE_DIFF('day', s.start_date, cn.cancellation_date))           AS fastest_cancel,
        MAX(DATE_DIFF('day', s.start_date, cn.cancellation_date))           AS slowest_cancel
    FROM cancellations cn
    JOIN subscriptions s ON cn.subscription_id = s.subscription_id
    GROUP BY s.plan_name
    ORDER BY avg_days_before_cancel
""").df()

,plan_name,avg_days_before_cancel,fastest_cancel,slowest_cancel
0,Starter,45.0,30,60
1,Regular,55.0,45,60
2,Full,62.0,40,90


In [13]:
conn.execute("""
    SELECT
        DATE_TRUNC('month', order_date) AS order_month,
        COUNT(*)                         AS total_orders,
        COUNT(*) FILTER (WHERE delivery_status = 'delayed') AS delayed,
        ROUND(COUNT(*) FILTER (WHERE delivery_status = 'delayed') * 100.0 /
              COUNT(*), 1)               AS delay_rate_pct
    FROM orders
    GROUP BY order_month
    ORDER BY order_month
""").df()

,order_month,total_orders,delayed,delay_rate_pct
0,2023-01-01,6,0,0.0
1,2023-02-01,10,1,10.0
2,2023-03-01,14,0,0.0
3,2023-04-01,19,0,0.0
4,2023-05-01,21,0,0.0
5,2023-06-01,22,0,0.0
6,2023-07-01,25,0,0.0
7,2023-08-01,26,0,0.0
8,2023-09-01,29,0,0.0
9,2023-10-01,30,0,0.0


In [14]:
conn.execute("""
    SELECT
        c.first_name,
        c.last_name,
        s.plan_name,
        o.order_date,
        o.delivery_status,
        cn.cancellation_date,
        cn.reason
    FROM orders o
    JOIN subscriptions s ON o.subscription_id = s.subscription_id
    JOIN customers c ON s.customer_id = c.customer_id
    LEFT JOIN cancellations cn ON s.subscription_id = cn.subscription_id
    WHERE o.delivery_status = 'delayed'
    ORDER BY s.plan_name, o.order_date
""").df()

,first_name,last_name,plan_name,order_date,delivery_status,cancellation_date,reason
0,Zara,Mitchell,Full,2024-02-01,delayed,NaT,NaN
1,Imogen,Evans,Full,2024-02-01,delayed,NaT,NaN
2,Casper,Baker,Full,2024-02-10,delayed,NaT,NaN
3,Freya,Young,Full,2024-02-12,delayed,NaT,NaN
4,Felix,Green,Full,2024-02-15,delayed,NaT,NaN
5,Scarlett,King,Full,2024-02-15,delayed,NaT,NaN
6,Violet,Turner,Full,2024-02-20,delayed,NaT,NaN
7,Willow,Scott,Full,2024-02-25,delayed,NaT,NaN
8,Jude,Wright,Full,2024-02-28,delayed,NaT,NaN
9,Theo,Allen,Regular,2024-02-01,delayed,NaT,NaN


In [15]:
conn.execute("""
    WITH cohort AS (
        SELECT
            c.customer_id,
            DATE_TRUNC('month', c.signup_date)        AS cohort_month,
            DATE_TRUNC('month', o.order_date)          AS order_month
        FROM customers c
            
        JOIN orders o ON c.customer_id = o.subscription_id
    ),
    cohort_size AS (
        SELECT cohort_month, COUNT(DISTINCT customer_id) AS total_customers
        FROM cohort
        GROUP BY cohort_month
    ),
    retention AS (
        SELECT
            cohort_month,
            order_month,
            COUNT(DISTINCT customer_id) AS active_customers
        FROM cohort
        GROUP BY cohort_month, order_month
    )
    SELECT
        r.cohort_month,
        r.order_month,
        cs.total_customers,
        r.active_customers,
        ROUND(r.active_customers * 100.0 / cs.total_customers, 0) AS retention_pct
    FROM retention r
    JOIN cohort_size cs ON r.cohort_month = cs.cohort_month
    ORDER BY r.cohort_month, r.order_month
""").df()

,cohort_month,order_month,total_customers,active_customers,retention_pct
0,2023-01-01,2023-01-01,6,6,100.0
1,2023-01-01,2023-02-01,6,5,83.0
2,2023-01-01,2023-03-01,6,5,83.0
3,2023-01-01,2023-04-01,6,4,67.0
4,2023-01-01,2023-05-01,6,4,67.0
...,...,...,...,...,...
105,2023-11-01,2023-11-01,1,1,100.0
106,2023-11-01,2023-12-01,1,1,100.0
107,2023-11-01,2024-01-01,1,1,100.0
108,2023-11-01,2024-02-01,1,1,100.0


In [16]:
conn.execute("""
    WITH cohort AS (
        SELECT
            c.customer_id,
            DATE_TRUNC('month', c.signup_date) AS cohort_month,
            DATEDIFF('month',
                DATE_TRUNC('month', c.signup_date),
                DATE_TRUNC('month', o.order_date)
            ) AS month_number
        FROM customers c
        JOIN orders o ON c.customer_id = o.subscription_id
    ),
    cohort_size AS (
        SELECT cohort_month, COUNT(DISTINCT customer_id) AS total_customers
        FROM cohort
        GROUP BY cohort_month
    ),
    retention AS (
        SELECT
            cohort_month,
            month_number,
            COUNT(DISTINCT customer_id) AS active_customers
        FROM cohort
        GROUP BY cohort_month, month_number
    )
    SELECT
        r.cohort_month,
        cs.total_customers,
        ROUND(MAX(CASE WHEN month_number = 0  THEN r.active_customers * 100.0 / cs.total_customers END), 0) AS m0,
        ROUND(MAX(CASE WHEN month_number = 1  THEN r.active_customers * 100.0 / cs.total_customers END), 0) AS m1,
        ROUND(MAX(CASE WHEN month_number = 2  THEN r.active_customers * 100.0 / cs.total_customers END), 0) AS m2,
        ROUND(MAX(CASE WHEN month_number = 3  THEN r.active_customers * 100.0 / cs.total_customers END), 0) AS m3,
        ROUND(MAX(CASE WHEN month_number = 4  THEN r.active_customers * 100.0 / cs.total_customers END), 0) AS m4,
        ROUND(MAX(CASE WHEN month_number = 5  THEN r.active_customers * 100.0 / cs.total_customers END), 0) AS m5,
        ROUND(MAX(CASE WHEN month_number = 6  THEN r.active_customers * 100.0 / cs.total_customers END), 0) AS m6
    FROM retention r
    JOIN cohort_size cs ON r.cohort_month = cs.cohort_month
    GROUP BY r.cohort_month, cs.total_customers
    ORDER BY r.cohort_month
""").df()

,cohort_month,total_customers,m0,m1,m2,m3,m4,m5,m6
0,2023-01-01,6,100.0,83.0,83.0,67.0,67.0,67.0,67.0
1,2023-02-01,5,100.0,100.0,100.0,80.0,80.0,80.0,80.0
2,2023-03-01,4,100.0,100.0,100.0,100.0,100.0,100.0,100.0
3,2023-04-01,6,100.0,100.0,50.0,50.0,50.0,50.0,50.0
4,2023-05-01,3,100.0,100.0,100.0,100.0,100.0,100.0,100.0
5,2023-06-01,4,100.0,100.0,50.0,50.0,50.0,50.0,50.0
6,2023-07-01,3,100.0,100.0,100.0,100.0,100.0,100.0,100.0
7,2023-08-01,3,100.0,100.0,67.0,67.0,67.0,67.0,67.0
8,2023-09-01,3,100.0,100.0,67.0,67.0,67.0,67.0,67.0
9,2023-10-01,2,100.0,100.0,50.0,50.0,50.0,50.0,NaN


In [17]:
conn.execute("""
    SELECT
        DATE_TRUNC('month', order_date) AS revenue_month,
        COUNT(DISTINCT subscription_id)  AS active_subscriptions,
        SUM(order_value)                 AS monthly_revenue
    FROM orders
    GROUP BY revenue_month
    ORDER BY revenue_month
""").df()

,revenue_month,active_subscriptions,monthly_revenue
0,2023-01-01,6,954.0
1,2023-02-01,10,1583.0
2,2023-03-01,14,2277.0
3,2023-04-01,19,3014.0
4,2023-05-01,21,3339.0
5,2023-06-01,22,3491.0
6,2023-07-01,25,3968.0
7,2023-08-01,26,4076.0
8,2023-09-01,29,4553.0
9,2023-10-01,30,4814.0


In [18]:
conn.execute("""
    WITH monthly_revenue AS (
        SELECT
            DATE_TRUNC('month', order_date) AS revenue_month,
            SUM(order_value)                AS monthly_revenue
        FROM orders
        GROUP BY revenue_month
    ),
    with_lag AS (
        SELECT
            revenue_month,
            monthly_revenue,
            LAG(monthly_revenue) OVER (ORDER BY revenue_month) AS prev_month_revenue
        FROM monthly_revenue
    )
    SELECT
        revenue_month,
        monthly_revenue,
        prev_month_revenue,
        ROUND(
            (monthly_revenue - prev_month_revenue) * 100.0 / prev_month_revenue
        , 1) AS mom_growth_pct
    FROM with_lag
    ORDER BY revenue_month
""").df()

,revenue_month,monthly_revenue,prev_month_revenue,mom_growth_pct
0,2023-01-01,954.0,NaN,NaN
1,2023-02-01,1583.0,954.0,65.9
2,2023-03-01,2277.0,1583.0,43.8
3,2023-04-01,3014.0,2277.0,32.4
4,2023-05-01,3339.0,3014.0,10.8
5,2023-06-01,3491.0,3339.0,4.6
6,2023-07-01,3968.0,3491.0,13.7
7,2023-08-01,4076.0,3968.0,2.7
8,2023-09-01,4553.0,4076.0,11.7
9,2023-10-01,4814.0,4553.0,5.7


In [19]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [20]:
df = conn.execute("""
    WITH monthly AS (
        SELECT
            DATE_TRUNC('month', o.order_date) AS revenue_month,
            SUM(o.order_value)                AS monthly_revenue,
            COUNT(DISTINCT o.subscription_id) AS active_subs
        FROM orders o
        GROUP BY revenue_month
    ),
    with_lag AS (
        SELECT
            revenue_month,
            monthly_revenue,
            active_subs,
            LAG(monthly_revenue) OVER (ORDER BY revenue_month) AS prev_revenue,
            LAG(active_subs)     OVER (ORDER BY revenue_month) AS prev_subs
        FROM monthly
    ),
    cancels AS (
        SELECT
            DATE_TRUNC('month', cancellation_date) AS cancel_month,
            COUNT(*)                                AS churned
        FROM cancellations
        GROUP BY cancel_month
    ),
    new_subs AS (
        SELECT
            DATE_TRUNC('month', start_date) AS start_month,
            COUNT(*)                         AS new_customers
        FROM subscriptions
        GROUP BY start_month
    )
    SELECT
        w.revenue_month,
        w.monthly_revenue,
        w.active_subs,
        COALESCE(n.new_customers, 0)                                    AS new_customers,
        COALESCE(c.churned, 0)                                          AS churned,
        ROUND((w.monthly_revenue - w.prev_revenue) * 100.0 
              / w.prev_revenue, 1)                                      AS mom_growth_pct
    FROM with_lag w
    LEFT JOIN cancels c  ON w.revenue_month = c.cancel_month
    LEFT JOIN new_subs n ON w.revenue_month = n.start_month
    ORDER BY w.revenue_month
""").df()

df

,revenue_month,monthly_revenue,active_subs,new_customers,churned,mom_growth_pct
0,2023-01-01,954.0,6,6,0,NaN
1,2023-02-01,1583.0,10,5,1,65.9
2,2023-03-01,2277.0,14,4,1,43.8
3,2023-04-01,3014.0,19,6,1,32.4
4,2023-05-01,3339.0,21,3,0,10.8
5,2023-06-01,3491.0,22,4,3,4.6
6,2023-07-01,3968.0,25,3,0,13.7
7,2023-08-01,4076.0,26,3,2,2.7
8,2023-09-01,4553.0,29,3,0,11.7
9,2023-10-01,4814.0,30,2,1,5.7


In [21]:
print(df[['revenue_month', 'active_subs', 'new_customers', 'churned']].to_string(index=False))

revenue_month  active_subs  new_customers  churned
   2023-01-01            6              6        0
   2023-02-01           10              5        1
   2023-03-01           14              4        1
   2023-04-01           19              6        1
   2023-05-01           21              3        0
   2023-06-01           22              4        3
   2023-07-01           25              3        0
   2023-08-01           26              3        2
   2023-09-01           29              3        0
   2023-10-01           30              2        1
   2023-11-01           30              1        1
   2023-12-01           29              0        1
   2024-01-01           29              0        0
   2024-02-01           29              0        0
   2024-03-01           29              0        0


In [22]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('MoM Revenue Growth %', 'Monthly Revenue (£)',
                    'New Subscriptions', 'Churned Subscriptions'),
    vertical_spacing=0.18,
    horizontal_spacing=0.12
)

months = df['revenue_month'].astype(str).str[:7]

fig.add_trace(
    go.Bar(
        x=months,
        y=df['mom_growth_pct'],
        marker_color=['#e74c3c' if v < 0 else '#2ecc71' 
                      for v in df['mom_growth_pct'].fillna(0)],
        name='MoM Growth %'
    ), row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=months,
        y=df['monthly_revenue'],
        mode='lines+markers',
        line=dict(color='#3498db', width=2),
        marker=dict(size=6),
        name='Monthly Revenue'
    ), row=1, col=2
)

fig.add_trace(
    go.Bar(
        x=months,
        y=df['new_customers'],
        marker_color='#3498db',
        name='New Customers'
    ), row=2, col=1
)

fig.add_trace(
    go.Bar(
        x=months,
        y=df['churned'],
        marker_color='#e74c3c',
        name='Churned'
    ), row=2, col=2
)

fig.update_layout(
    title=dict(
        text='Butternut Box — Revenue Health Dashboard',
        font=dict(size=18)
    ),
    height=600,
    showlegend=False,
    plot_bgcolor='#FFFFFF',
    paper_bgcolor='#FFFFFF'
)
# #fff6d8
fig.update_xaxes(tickangle=45)
fig.show()

In [23]:
latest = df.dropna(subset=['mom_growth_pct']).iloc[-1]
month = str(latest['revenue_month'])[:7]
mom = latest['mom_growth_pct']
revenue = latest['monthly_revenue']

color = '#2ecc71' if mom >= 0 else '#e74c3c'
arrow = '▲' if mom >= 0 else '▼'

print(f"BUTTERNUT BOX — REVENUE HEALTH")
print(f"{'─' * 35}")
print(f"Latest month      : {month}")
print(f"Monthly revenue   : £{revenue:,.0f}")
print(f"MoM growth        : {arrow} {mom}%")
print(f"{'─' * 35}")

BUTTERNUT BOX — REVENUE HEALTH
───────────────────────────────────
Latest month      : 2024-03
Monthly revenue   : £4,597
MoM growth        : ▲ 0.0%
───────────────────────────────────


In [24]:
print("=" * 50)
print("BUTTERNUT BOX — DATA QUALITY REPORT")
print("=" * 50)

BUTTERNUT BOX — DATA QUALITY REPORT


In [25]:
null_check = conn.execute("""
    SELECT 'customers' AS table_name, 'customer_id'   AS column_name, COUNT(*) FILTER (WHERE customer_id   IS NULL) AS null_count FROM customers UNION ALL
    SELECT 'customers', 'first_name',                                  COUNT(*) FILTER (WHERE first_name    IS NULL) FROM customers UNION ALL
    SELECT 'customers', 'last_name',                                   COUNT(*) FILTER (WHERE last_name     IS NULL) FROM customers UNION ALL
    SELECT 'customers', 'email',                                       COUNT(*) FILTER (WHERE email         IS NULL) FROM customers UNION ALL
    SELECT 'customers', 'dog_size',                                    COUNT(*) FILTER (WHERE dog_size      IS NULL) FROM customers UNION ALL
    SELECT 'customers', 'signup_date',                                 COUNT(*) FILTER (WHERE signup_date   IS NULL) FROM customers UNION ALL
    SELECT 'subscriptions', 'customer_id',                             COUNT(*) FILTER (WHERE customer_id   IS NULL) FROM subscriptions UNION ALL
    SELECT 'subscriptions', 'plan_name',                               COUNT(*) FILTER (WHERE plan_name     IS NULL) FROM subscriptions UNION ALL
    SELECT 'subscriptions', 'status',                                  COUNT(*) FILTER (WHERE status        IS NULL) FROM subscriptions UNION ALL
    SELECT 'orders', 'order_date',                                     COUNT(*) FILTER (WHERE order_date    IS NULL) FROM orders UNION ALL
    SELECT 'orders', 'delivery_date',                                  COUNT(*) FILTER (WHERE delivery_date IS NULL) FROM orders UNION ALL
    SELECT 'orders', 'order_value',                                    COUNT(*) FILTER (WHERE order_value   IS NULL) FROM orders UNION ALL
    SELECT 'cancellations', 'cancellation_date',                       COUNT(*) FILTER (WHERE cancellation_date IS NULL) FROM cancellations UNION ALL
    SELECT 'cancellations', 'reason',                                  COUNT(*) FILTER (WHERE reason        IS NULL) FROM cancellations
""").df()

null_check['status'] = null_check['null_count'].apply(lambda x: '✅ PASS' if x == 0 else '❌ FAIL')
print(null_check.to_string(index=False))

   table_name       column_name  null_count status
    customers       customer_id           0 ✅ PASS
    customers        first_name           0 ✅ PASS
    customers         last_name           0 ✅ PASS
    customers             email           0 ✅ PASS
    customers          dog_size           0 ✅ PASS
    customers       signup_date           0 ✅ PASS
subscriptions       customer_id           0 ✅ PASS
subscriptions         plan_name           0 ✅ PASS
subscriptions            status           0 ✅ PASS
       orders        order_date           0 ✅ PASS
       orders     delivery_date           0 ✅ PASS
       orders       order_value           0 ✅ PASS
cancellations cancellation_date           0 ✅ PASS
cancellations            reason           0 ✅ PASS


In [26]:
duplicate_check = conn.execute("""
    SELECT 'customers' AS table_name, 
            'customer_id' AS id_column, 
            COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_count 
            FROM customers UNION ALL
    SELECT 'subscriptions', 
            'subscription_id', 
            COUNT(*) - COUNT(DISTINCT subscription_id) 
            FROM subscriptions UNION ALL
    SELECT 'orders', 'order_id', COUNT(*) - COUNT(DISTINCT order_id) 
            FROM orders UNION ALL
    SELECT 'cancellations', 
            'cancellation_id', 
            COUNT(*) - COUNT(DISTINCT cancellation_id) 
            FROM cancellations
""").df()

duplicate_check['status'] = duplicate_check['duplicate_count'].apply(lambda x: '✅ PASS' if x == 0 else '❌ FAIL')
print(duplicate_check.to_string(index=False))

   table_name       id_column  duplicate_count status
    customers     customer_id                0 ✅ PASS
subscriptions subscription_id                0 ✅ PASS
       orders        order_id                0 ✅ PASS
cancellations cancellation_id                0 ✅ PASS


In [27]:
integrity_check = conn.execute("""
    SELECT 'orders -> subscriptions' AS relationship,
           'orphaned orders'         AS check_type,
           COUNT(*)                  AS violation_count
    FROM orders o
    LEFT JOIN subscriptions s ON o.subscription_id = s.subscription_id
    WHERE s.subscription_id IS NULL

    UNION ALL

    SELECT 'cancellations -> subscriptions',
           'orphaned cancellations',
           COUNT(*)
    FROM cancellations c
    LEFT JOIN subscriptions s ON c.subscription_id = s.subscription_id
    WHERE s.subscription_id IS NULL

    UNION ALL

    SELECT 'subscriptions -> customers',
           'orphaned subscriptions',
           COUNT(*)
    FROM subscriptions s
    LEFT JOIN customers c ON s.customer_id = c.customer_id
    WHERE c.customer_id IS NULL

    UNION ALL

    SELECT 'orders',
           'delivery before order date',
           COUNT(*)
    FROM orders
    WHERE delivery_date < order_date

    UNION ALL

    SELECT 'orders vs subscriptions',
           'order before signup date',
           COUNT(*)
    FROM orders o
    JOIN subscriptions s ON o.subscription_id = s.subscription_id
    JOIN customers c ON s.customer_id = c.customer_id
    WHERE o.order_date < c.signup_date

    UNION ALL

    SELECT 'cancellations vs subscriptions',
           'cancellation before signup',
           COUNT(*)
    FROM cancellations c
    JOIN subscriptions s ON c.subscription_id = s.subscription_id
    WHERE c.cancellation_date < s.start_date
""").df()

integrity_check['status'] = integrity_check['violation_count'].apply(lambda x: '✅ PASS' if x == 0 else '❌ FAIL')
print(integrity_check.to_string(index=False))

                  relationship                 check_type  violation_count status
       orders -> subscriptions            orphaned orders                0 ✅ PASS
cancellations -> subscriptions     orphaned cancellations                0 ✅ PASS
    subscriptions -> customers     orphaned subscriptions                0 ✅ PASS
                        orders delivery before order date                0 ✅ PASS
       orders vs subscriptions   order before signup date                0 ✅ PASS
cancellations vs subscriptions cancellation before signup                0 ✅ PASS


In [28]:
logic_check = conn.execute("""
    SELECT 'orders'         AS table_name,
           'negative or zero order value' AS check_type,
           COUNT(*)         AS violation_count
    FROM orders
    WHERE order_value <= 0

    UNION ALL

    SELECT 'subscriptions',
           'negative or zero weekly price',
           COUNT(*)
    FROM subscriptions
    WHERE weekly_price <= 0

    UNION ALL

    SELECT 'subscriptions',
           'plan price mismatch',
           COUNT(*)
    FROM subscriptions
    WHERE (plan_name = 'Starter' AND weekly_price != 25.00)
       OR (plan_name = 'Regular' AND weekly_price != 35.00)
       OR (plan_name = 'Full'    AND weekly_price != 50.00)

    UNION ALL

    SELECT 'orders',
           'order value mismatch for plan',
           COUNT(*)
    FROM orders o
    JOIN subscriptions s ON o.subscription_id = s.subscription_id
    WHERE (s.plan_name = 'Starter' AND o.order_value != 108.00)
       OR (s.plan_name = 'Regular' AND o.order_value != 152.00)
       OR (s.plan_name = 'Full'    AND o.order_value != 217.00)

    UNION ALL

    SELECT 'cancellations',
           'cancellation on active subscription',
           COUNT(*)
    FROM cancellations c
    JOIN subscriptions s ON c.subscription_id = s.subscription_id
    WHERE s.status = 'active'

    UNION ALL

    SELECT 'cancellations',
           'multiple cancellations per subscription',
           COUNT(*) - COUNT(DISTINCT subscription_id)
    FROM cancellations
""").df()

logic_check['status'] = logic_check['violation_count'].apply(lambda x: '✅ PASS' if x == 0 else '❌ FAIL')
print(logic_check.to_string(index=False))

   table_name                              check_type  violation_count status
       orders            negative or zero order value                0 ✅ PASS
subscriptions           negative or zero weekly price                0 ✅ PASS
subscriptions                     plan price mismatch                0 ✅ PASS
       orders           order value mismatch for plan                0 ✅ PASS
cancellations     cancellation on active subscription                0 ✅ PASS
cancellations multiple cancellations per subscription                0 ✅ PASS


In [29]:
print("=" * 55)
print("BUTTERNUT BOX — FULL DATA QUALITY REPORT")
print("=" * 55)

checks = {
    "1. Null Check":        null_check,
    "2. Duplicate Check":   duplicate_check,
    "3. Integrity Check":   integrity_check,
    "4. Business Logic":    logic_check
}

total_pass = 0
total_fail = 0

for section, df in checks.items():
    passes = (df['status'] == '✅ PASS').sum()
    fails  = (df['status'] == '❌ FAIL').sum()
    total_pass += passes
    total_fail += fails
    print(f"\n{section} — {passes} passed, {fails} failed")

print("\n" + "=" * 55)
print(f"TOTAL: {total_pass} passed, {total_fail} failed")
status = "✅ DATABASE CERTIFIED CLEAN" if total_fail == 0 else "❌ DATABASE HAS ISSUES — DO NOT ANALYSE"
print(status)
print("=" * 55)

BUTTERNUT BOX — FULL DATA QUALITY REPORT

1. Null Check — 14 passed, 0 failed

2. Duplicate Check — 4 passed, 0 failed

3. Integrity Check — 6 passed, 0 failed

4. Business Logic — 6 passed, 0 failed

TOTAL: 30 passed, 0 failed
✅ DATABASE CERTIFIED CLEAN


In [30]:
conn.execute("""
    SELECT
        c.first_name,
        c.dog_name,
        pp.weight_kg,
        pp.health_condition
    FROM customers c
    JOIN pet_profiles pp ON c.customer_id = pp.customer_id
    LIMIT 5
""").df()

,first_name,dog_name,weight_kg,health_condition
0,Emma,Bella,6.6,joint problems
1,James,Coco,5.2,none
2,Sophie,Pip,11.4,none
3,Oliver,Milo,10.1,skin issues
4,Amelia,Rosie,4.4,skin issues


In [31]:
conn.execute("""
    SELECT
        c.first_name,
        c.dog_size,
        pp.weight_kg,
        pp.health_condition,
        pp.previous_food_type,
        r.recipe_name,
        oi.pouches
    FROM customers c
    JOIN pet_profiles pp  ON c.customer_id = pp.customer_id
    JOIN orders o         ON c.customer_id = o.subscription_id
    JOIN order_items oi   ON o.order_id = oi.order_id
    JOIN recipes r        ON oi.recipe_id = r.recipe_id
    LIMIT 10
""").df()

,first_name,dog_size,weight_kg,health_condition,previous_food_type,recipe_name,pouches
0,Emma,small,6.6,joint problems,dry kibble,Pork This Way,8
1,Emma,small,6.6,joint problems,dry kibble,Gobble Gobble Turkey,8
2,Emma,small,6.6,joint problems,dry kibble,Chicken You Out,8
3,Emma,small,6.6,joint problems,dry kibble,You've Got Game,8
4,Emma,small,6.6,joint problems,dry kibble,Duo of Duck & Chicken,8
5,Emma,small,6.6,joint problems,dry kibble,Pork This Way,8
6,Emma,small,6.6,joint problems,dry kibble,Gobble Gobble Turkey,8
7,Emma,small,6.6,joint problems,dry kibble,Chicken You Out,8
8,Emma,small,6.6,joint problems,dry kibble,You've Got Game,8
9,Emma,small,6.6,joint problems,dry kibble,Duo of Duck & Chicken,8


In [32]:
conn.execute("""
    SELECT
        c.first_name,
        c.dog_size,
        pp.weight_kg,
        pp.health_condition,
        pp.previous_food_type,
        r.recipe_name,
        oi.pouches
    FROM customers c
    JOIN pet_profiles pp  ON c.customer_id = pp.customer_id
    JOIN orders o         ON c.customer_id = o.subscription_id
    JOIN order_items oi   ON o.order_id = oi.order_id
    JOIN recipes r        ON oi.recipe_id = r.recipe_id
    LIMIT 10
""").df()

,first_name,dog_size,weight_kg,health_condition,previous_food_type,recipe_name,pouches
0,Emma,small,6.6,joint problems,dry kibble,Pork This Way,8
1,Emma,small,6.6,joint problems,dry kibble,Gobble Gobble Turkey,8
2,Emma,small,6.6,joint problems,dry kibble,Chicken You Out,8
3,Emma,small,6.6,joint problems,dry kibble,You've Got Game,8
4,Emma,small,6.6,joint problems,dry kibble,Duo of Duck & Chicken,8
5,Emma,small,6.6,joint problems,dry kibble,Pork This Way,8
6,Emma,small,6.6,joint problems,dry kibble,Gobble Gobble Turkey,8
7,Emma,small,6.6,joint problems,dry kibble,Chicken You Out,8
8,Emma,small,6.6,joint problems,dry kibble,You've Got Game,8
9,Emma,small,6.6,joint problems,dry kibble,Duo of Duck & Chicken,8


In [33]:
conn.execute("""
    SELECT
        o.subscription_id,
        COUNT(DISTINCT oi.recipe_id) AS distinct_recipes
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    GROUP BY o.subscription_id
""").df()

,subscription_id,distinct_recipes
0,11,1
1,30,4
2,40,3
3,29,3
4,35,1
5,3,1
6,19,1
7,36,4
8,38,1
9,6,4


In [34]:
conn.execute("""
    SELECT
        o.subscription_id,
        COUNT(DISTINCT oi.recipe_id) AS distinct_recipes
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    GROUP BY o.subscription_id
""").df()

,subscription_id,distinct_recipes
0,2,4
1,26,3
2,22,1
3,5,3
4,24,5
5,25,5
6,34,3
7,11,1
8,30,4
9,19,1


In [35]:
conn.execute("""
    WITH recipe_engagement AS (
        SELECT
            o.subscription_id,
            COUNT(DISTINCT oi.recipe_id) AS distinct_recipes
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        GROUP BY o.subscription_id
    )
    SELECT
        s.status,
        re.subscription_id,
        re.distinct_recipes
    FROM recipe_engagement re
    JOIN subscriptions s ON re.subscription_id = s.subscription_id
    ORDER BY s.status, re.distinct_recipes
""").df()

,status,subscription_id,distinct_recipes
0,active,37,3
1,active,5,3
2,active,34,3
3,active,31,3
4,active,12,3
5,active,13,3
6,active,14,3
7,active,16,3
8,active,17,3
9,active,29,3


In [36]:
conn.execute("""
    WITH recipe_engagement AS (
        SELECT
            o.subscription_id,
            COUNT(DISTINCT oi.recipe_id) AS distinct_recipes
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        GROUP BY o.subscription_id
    )
    SELECT
        s.status,
        ROUND(AVG(re.distinct_recipes), 1) AS avg_distinct_recipes,
        MIN(re.distinct_recipes)            AS min_recipes,
        MAX(re.distinct_recipes)            AS max_recipes
    FROM recipe_engagement re
    JOIN subscriptions s ON re.subscription_id = s.subscription_id
    GROUP BY s.status
""").df()

,status,avg_distinct_recipes,min_recipes,max_recipes
0,active,3.9,3,5
1,cancelled,1.0,1,1


In [37]:
conn.execute("""
    SELECT
        pp.health_condition,
        s.status
    FROM pet_profiles pp
    JOIN subscriptions s ON pp.customer_id = s.customer_id
""").df()

,health_condition,status
0,joint problems,active
1,none,active
2,none,cancelled
3,skin issues,active
4,skin issues,active
5,weight management,active
6,none,cancelled
7,joint problems,active
8,joint problems,active
9,weight management,active


In [38]:
conn.execute("""
    SELECT
        pp.health_condition,
        COUNT(*) FILTER (WHERE s.status = 'active')    AS active_customers,
        COUNT(*) FILTER (WHERE s.status = 'cancelled') AS cancelled_customers
    FROM pet_profiles pp
    JOIN subscriptions s ON pp.customer_id = s.customer_id
    GROUP BY pp.health_condition
    ORDER BY pp.health_condition
""").df()

,health_condition,active_customers,cancelled_customers
0,joint problems,5,0
1,none,10,9
2,sensitive stomach,4,0
3,skin issues,6,0
4,weight management,4,2


In [41]:
conn.close()